In [ ]:
!pip install ipython==8.12.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 796.4/796.4 kB 50.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 94.8 MB/s eta 0:00:00
  Attempting uninstall: ipython
    Found existing installation: ipython 7.34.0
    Uninstalling ipython-7.34.0:
      Successfully uninstalled ipython-7.34.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires ipython==7.34.0, but you have ipython 8.12.0 which is incompatible.


In [ ]:
!pip install datasets
!pip install einops
!pip install transformers
!pip install evaluate


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.6 MB/s eta 0:00:00


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset


import transformers
from einops import rearrange
from einops.layers.torch import Rearrange
from datasets import load_dataset

from transformers import GPT2Tokenizer
from transformers import AutoTokenizer
from transformers import AutoModelForCausalLM
from transformers import TrainingArguments
from transformers import Trainer

import os
import sys
import json
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import evaluate

from google.colab import drive

# torch.manual_seed(42)
torch.manual_seed(110)

In [ ]:
drive.mount('/content/gdrive')

# NOTE: Make sure your path does NOT include a '/' at the end!
base_dir = "/content/gdrive/MyDrive/CS4782 Deep Learning Project/lora-code"
sys.path.append(base_dir)


Mounted at /content/gdrive


In [ ]:
# This makes sure the submission module is reloaded whenever you make edits.
%load_ext autoreload
%aimport lora_model
%autoreload 1
import lora_model

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(F"Device set to {device}")


Device set to cuda


In [ ]:
bleu = evaluate.load("bleu")
meteor = evaluate.load("meteor")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


In [ ]:
from utils import count_trainable_params

Prepare Dataset

In [ ]:
# Load SST-2

DATA_DIR = base_dir + "/data/e2e-dataset/"
train_df = pd.read_csv(os.path.join(DATA_DIR, "trainset.csv"))
val_df   = pd.read_csv(os.path.join(DATA_DIR, "devset.csv"))
testwref_df  = pd.read_csv(os.path.join(DATA_DIR, "testset_w_refs.csv"))
test_df = pd.read_csv(os.path.join(DATA_DIR, "testset.csv"))
print(train_df.head())


# print(e2e_data)

                                                  mr  \
0  name[The Vaults], eatType[pub], priceRange[mor...   
1  name[The Cambridge Blue], eatType[pub], food[E...   
2  name[The Eagle], eatType[coffee shop], food[Ja...   
3  name[The Mill], eatType[coffee shop], food[Fre...   
4  name[Loch Fyne], food[French], customer rating...   

                                                 ref  
0  The Vaults pub near Café Adriatic has a 5 star...  
1  Close to Café Brazil, The Cambridge Blue pub s...  
2  The Eagle is a low rated coffee shop near Burg...  
3  Located near The Sorrento is a French Theme ea...  
4  For luxurious French food, the Loch Fyne is lo...  


In [ ]:
train_df = train_df.rename(columns={"mr":"text", "ref":"label"})
val_df = val_df.rename(columns={"mr":"text", "ref":"label"})
testwref_df = testwref_df.rename(columns={"mr":"text", "ref":"label"})
test_df = test_df.rename(columns={"MR":"text"})

print(train_df.columns.tolist())


print("Length before cleaning: ", len(train_df))


def clean_dataframe(df):
    df = df.dropna()
    df = df[df["text"].str.len() > 0]
    cols = df.columns.to_list()
    if "label" in cols:
        df = df[df["label"].str.len() > 0]
    return df

train_df = clean_dataframe(train_df)
val_df = clean_dataframe(val_df)
test_df = clean_dataframe(test_df)
testwref_df = clean_dataframe(testwref_df)
print("Length after cleaning: ", len(train_df))

print(train_df.head())
print(test_df.head())


['text', 'label']
Length before cleaning:  42061
Length after cleaning:  42061
                                                text  \
0  name[The Vaults], eatType[pub], priceRange[mor...   
1  name[The Cambridge Blue], eatType[pub], food[E...   
2  name[The Eagle], eatType[coffee shop], food[Ja...   
3  name[The Mill], eatType[coffee shop], food[Fre...   
4  name[Loch Fyne], food[French], customer rating...   

                                               label  
0  The Vaults pub near Café Adriatic has a 5 star...  
1  Close to Café Brazil, The Cambridge Blue pub s...  
2  The Eagle is a low rated coffee shop near Burg...  
3  Located near The Sorrento is a French Theme ea...  
4  For luxurious French food, the Loch Fyne is lo...  
                                                text
0  name[Blue Spice], eatType[coffee shop], area[c...
1  name[Blue Spice], eatType[coffee shop], area[r...
2  name[Blue Spice], eatType[coffee shop], custom...
3  name[Blue Spice], eatType[coffee shop],

Tokenizer

In [ ]:
# tokenizer = AutoTokenizer.from_pretrained('gpt2-medium')
# # tokenizer = AutoTokenizer.from_pretrained('gpt2-medium')
# tokenizer.add_special_tokens({'pad_token': '[PAD]'})
# tokenizer.SPECIAL_TOKENS_ATTRIBUTES.append("delimiter")
# tokenizer.add_special_tokens({'delimiter': '||'})
# MAX_LEN = 128

DELIMITER = "||"
MAX_LEN = 512

# tokenizer = AutoTokenizer.from_pretrained('gpt2-medium')
def get_tokenizer():
    tokenizer = AutoTokenizer.from_pretrained('gpt2-medium')

    print("Tokenizer token len - before: ", len(tokenizer))
    # tokenizer = AutoTokenizer.from_pretrained('gpt2-medium')
    tokenizer.add_special_tokens({'pad_token': '[PAD]'})
    tokenizer.pad_token_id = tokenizer.convert_tokens_to_ids("[PAD]")
    # tokenizer.SPECIAL_TOKENS_ATTRIBUTES.append("delimiter")
    # tokenizer.add_special_tokens({'delimiter': '||'})
    tokenizer.add_tokens([DELIMITER.strip()])   # adds "||" as one token
    print("Tokenizer token len - after: ", len(tokenizer))
    return tokenizer

tokenizer = get_tokenizer()



def tokenize_text(text,padding='max_length'):
    tokens = tokenizer(text, padding=padding, truncation=True, return_tensors="pt", max_length=MAX_LEN)
    return tokens


Tokenizer token len - before:  50257
Tokenizer token len - after:  50258


Custom DataSet

In [ ]:
tokenizer.pad_token_id

50257

In [ ]:
class E2EDataset(Dataset):
    def __init__(self, dataframe, test=False):
        self.data = dataframe.reset_index(drop=True)
        self.test = test

    def __len__(self):
        '''
        Returns the length of the dataset.
        '''
        return len(self.data)

    def __getitem__(self, idx):
        '''
        Returns the training/validation/test example as index idx.
        '''

        row = self.data.iloc[idx]
        text = row["text"].strip()
        combined = text
        if len(row) > 1:
            label = row["label"].strip()
            combined = text + " " + DELIMITER + " " + label

        if not self.test:
            tokens = tokenize_text(combined)
            input_ids = tokens["input_ids"].squeeze()
            attention_mask = tokens["attention_mask"].squeeze()
            inputs = {
                'input_ids' : input_ids.to(dtype=torch.long),
                'attention_mask' : attention_mask.to(dtype=torch.long),
            }
            inputs['labels'] = inputs['input_ids'].clone()

        else:   # for generation
            tokens = tokenize_text(text)
            input_ids = tokens["input_ids"].squeeze()
            attention_mask = tokens["attention_mask"].squeeze()
            inputs = {
                'input_ids' : input_ids.to(dtype=torch.long),
                'attention_mask' : attention_mask.to(dtype=torch.long),
            }
            label_tokens = tokenizer(label, padding='max_length', truncation=True, return_tensors="pt", max_length=MAX_LEN)
            label_ids = label_tokens["input_ids"].squeeze()
            inputs["labels"] = label_ids
            inputs['prompt_len'] = len(text)

        return inputs


train_dataset = E2EDataset(train_df)
val_dataset = E2EDataset(val_df)
test_dataset = E2EDataset(test_df)

In [ ]:
BATCH_SIZE = 8

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

# batch = next(iter(train_loader))
# print(batch["input_ids"].shape)
# print(batch["labels"].shape)

Model

In [ ]:
model = AutoModelForCausalLM.from_pretrained("gpt2-medium").to(device)
with torch.no_grad():
    model.resize_token_embeddings(len(tokenizer))  # TODO: remember to add this after adding pad token to tokenizer
model.config.pad_token_id = tokenizer.pad_token_id   # TODO: remember to add this after adding pad token to tokenizer


Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2-medium
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...23}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Training

In [ ]:
from transformers import TrainerCallback
from google.colab import auth
from googleapiclient.discovery import build


class CustomCallback(TrainerCallback):
    def __init__(self):
        auth.authenticate_user()
        self.drive_service = build('drive', 'v3')

    def on_epoch_end(self, args, state, control, **kwargs):
        self.drive_service.files().emptyTrash().execute()

custom_callback = CustomCallback()

In [ ]:
OUTPUT_DIR = base_dir + "/models/checkpoints-e2e/baseline_512seqlen/"

# Trainer by default uses AdamW
training_args = TrainingArguments(
    # output_dir=".",
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=4,
    num_train_epochs=5,
    learning_rate=0.0002,
    warmup_steps=500,
    lr_scheduler_type="linear",
    weight_decay=0.01,
    logging_steps=50,
    save_steps=500,
    label_smoothing_factor=0.1,
    adam_beta2=0.999,
    max_grad_norm=0.0,
    gradient_accumulation_steps=1,
    fp16=True,
    save_strategy = "epoch",
    save_total_limit=1,
    logging_strategy = "epoch",
    eval_strategy="epoch",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,   # your processed E2E dataset
    eval_dataset=val_dataset,
    # callbacks = [custom_callback]
)

existing_callbacks = trainer.callback_handler.callbacks
print(existing_callbacks)
trainer.callback_handler.callbacks.insert(0, custom_callback)
print(trainer.callback_handler.callbacks)


# trainer.train(resume_from_checkpoint=OUTPUT_DIR + "checkpoint-10516")
# trainer.train()

[<transformers.trainer_callback.DefaultFlowCallback object at 0x787ecc7974a0>, <transformers.utils.notebook.NotebookProgressCallback object at 0x78808422d2e0>]
[<__main__.CustomCallback object at 0x787ecc762120>, <transformers.trainer_callback.DefaultFlowCallback object at 0x787ecc7974a0>, <transformers.utils.notebook.NotebookProgressCallback object at 0x78808422d2e0>]


In [ ]:

# trainer.train(resume_from_checkpoint=OUTPUT_DIR + "checkpoint-10516")
trainer.train(resume_from_checkpoint=True)

There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


Epoch,Training Loss,Validation Loss
4,1.464093,1.490929
5,1.458080,1.492327


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=26290, training_loss=0.584434618973469, metrics={'train_runtime': 8114.1627, 'train_samples_per_second': 25.918, 'train_steps_per_second': 3.24, 'total_flos': 1.9531039956271104e+17, 'train_loss': 0.584434618973469, 'epoch': 5.0})

In [ ]:
# Save just weights
OUTPUT_DIR = base_dir + "/models/checkpoints-e2e/baseline_512seqlen/"

model = trainer.model
torch.save(model.state_dict(), OUTPUT_DIR + "model_weights.pt")


In [ ]:
# Load model from just weights
model = AutoModelForCausalLM.from_pretrained("gpt2-medium").to(device) # loads config + struct
tokenizer = get_tokenizer()
with torch.no_grad():
    model.resize_token_embeddings(len(tokenizer))
model.config.pad_token_id = tokenizer.pad_token_id

model.load_state_dict(torch.load(OUTPUT_DIR + "model_weights.pt"))

In [ ]:
model_ft_paramcount = count_trainable_params(model)
print(model_ft_paramcount)
print("{} M".format(model_ft_paramcount / 1000000))

354824192
354.824192 M


Evaluation

In [ ]:
class TestDataset(Dataset):
    def __init__(self, data: list):
        self.data = data

    def __len__(self):
        '''
        Returns the length of the dataset.
        '''
        return len(self.data)

    def __getitem__(self, idx):
        '''
        Returns the training/validation/test example as index idx.
        '''

        curr_data = self.data[idx]
        data_tokens = tokenize_text(curr_data, padding=False)
        input_ids = data_tokens["input_ids"].squeeze()
        attention_mask = data_tokens["attention_mask"].squeeze()
        # prompt_len = len(tokenizer.encode(curr_data, add_special_tokens=False,return_tensors="pt").squeeze())
        prompt_len = len(curr_data)

        inputs = {
            'input_ids' : input_ids.to(dtype=torch.long),
            'attention_mask' : attention_mask.to(dtype=torch.long),
            'prompt_len' : prompt_len
        }
        return inputs


In [ ]:
def data_retrieval(df, unique_mr=False):
        if unique_mr: # for bleu, each unique mr has a list of refs
            mr_ref_mapping = {}
            for idx in range(len(df)):
                row = df.iloc[idx]
                mr = row["text"].strip()
                ref = row["label"].strip()

                if mr not in mr_ref_mapping:
                    mr_ref_mapping[mr] = []
                mr_ref_mapping[mr].append(ref)

            mrs = list(mr_ref_mapping.keys())
            refs = list(mr_ref_mapping.values())

            return mrs, refs

        else:   # each mr has a ref, may have duplicate mrs
            mrs = []
            refs = []
            for idx in range(len(df)):
                row = df.iloc[idx]
                mr = row["text"].strip()
                ref = row["label"].strip()
                mrs.append(mr)
                refs.append(ref)

            return mrs, refs

In [ ]:
# generate only using MR
# compare against only ref

def batched_generate(model, tokenizer, dataset):
    model.eval()

    preds = []

    # dataloader = DataLoader(dataset, batch_size=BATCH_SIZE)
    dataloader = DataLoader(dataset, batch_size=1)


    for batch_idx, data in tqdm(enumerate(dataloader), total=len(dataloader)):
        mr_ids = data["input_ids"].to(device)
        mr_attention_mask = data["attention_mask"].to(device)

        with torch.no_grad():
            # output = model.generate(
            #     input_ids=mr_ids,
            #     attention_mask=mr_attention_mask,
            #     pad_token_id=tokenizer.pad_token_id,
            #     max_new_tokens=40
            # )

            output = model.generate(
                input_ids=mr_ids,
                attention_mask=mr_attention_mask,
                max_new_tokens=64,
                num_beams=10,
                length_penalty=0.9,
                no_repeat_ngram_size=4,
                repetition_penalty=1.0,
                early_stopping=True,
                pad_token_id=tokenizer.pad_token_id,
                # eos_token_id=628,
                eos_token_id=tokenizer.eos_token_id
            )


        # # SEPERATOR BASED MR REMOVAL ====================
        decoded = tokenizer.decode(output[0], skip_special_tokens=False)
        text = decoded
        text_cleaned = decoded

        # identify where prediction begins
        if DELIMITER in text_cleaned:
            text_cleaned = text_cleaned.split(DELIMITER, 1)[1].strip()

        # identify where prediction begins
        if DELIMITER in text_cleaned:
            text_cleaned = text_cleaned.replace(DELIMITER, "").strip()

        if tokenizer.pad_token in text_cleaned:
            text_cleaned = text_cleaned.replace(tokenizer.pad_token, "").strip()

        preds.append(text_cleaned)
        # ====================

        # # INPUT LENGTH BASED MR REMOVAL ====================
        # decoded = tokenizer.decode(output[0], skip_special_tokens=False)
        # prompt_len = data["prompt_len"]
        # text = decoded[prompt_len:]
        # text_cleaned = text
        # # if batch_idx == 0:
        # #     print("{}'th mr decoded prediction: {}".format(batch_idx, text_cleaned))

        # if DELIMITER in text_cleaned:
        #     text_cleaned = text_cleaned.split(DELIMITER, 1)[1].strip()

        # if tokenizer.pad_token in text_cleaned:
        #     text_cleaned = text_cleaned.replace(tokenizer.pad_token, "").strip()

        # # if batch_idx == 0:
        # #     print("{}'th mr decoded prediction: {}".format(batch_idx, text_cleaned))
        # preds.append(text_cleaned)
        # # ====================



        # print("BATCH: \n", batch_idx)
        # print("Raw Decoded: \n", decoded)
        # print("Trimmed: \n", text)
        # print("Cleaned: \n",text_cleaned)
        # print("Decoded MR1: \n",tokenizer.decode(mr_ids))
        # print("Decoded MR2: \n",tokenizer.decode(mr_ids,skip_special_tokens=True))

        # if batch_idx == 10:
        #     # print("Batch decode: ", decoded)
        #     # print("batch decode formatted", decoded_no_mr)
        #     # print("stripped_pad: " , decoded[0].replace(tokenizer.pad_token, "").strip())

        #     # print("BATCH: \n", batch_idx)
        #     # print("Raw Decoded: \n", decoded)
        #     # print("Trimmed: \n", text)
        #     # print("Cleaned: \n",text_cleaned)
        #     # print("Decoded MR1: \n",tokenizer.decode(mr_ids))
        #     # print("Decoded MR2: \n",tokenizer.decode(mr_ids,skip_special_tokens=True))

        #     break


    model.train()
    return preds

In [ ]:
raw_mrs, refs = data_retrieval(testwref_df, unique_mr=True)
raw_mrs_dataset = TestDataset(raw_mrs)
raw_mrs_dataset.__getitem__(0)

{'input_ids': tensor([ 3672,    58, 14573, 43537,  4357,  4483,  6030,    58,  1073,  5853,
          6128,  4357,  1989,    58, 19205,  7372,    60]),
 'attention_mask': tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]),
 'prompt_len': 57}

In [ ]:
raw_mrs, refs = data_retrieval(testwref_df, unique_mr=True)
raw_mrs_dataset = TestDataset(raw_mrs)
preds = batched_generate(model, tokenizer, raw_mrs_dataset)
print("1st MR's refs: ", refs[0])


  0%|          | 0/630 [00:00<?, ?it/s]

BATCH: 
 0
Raw Decoded: 
 name[Blue Spice], eatType[coffee shop], area[city centre] || Blue Spice is a coffee shop that is located in the city centre.  It has a price range of £20-25.[PAD][PAD][PAD][PAD]afé Sicilia.  Blue Spice is an average price range for a meal.  The Blue Spice coffee shop is an average priced coffee shop in the city
Trimmed: 
 name[Blue Spice], eatType[coffee shop], area[city centre] || Blue Spice is a coffee shop that is located in the city centre.  It has a price range of £20-25.[PAD][PAD][PAD][PAD]afé Sicilia.  Blue Spice is an average price range for a meal.  The Blue Spice coffee shop is an average priced coffee shop in the city
Cleaned: 
 Blue Spice is a coffee shop that is located in the city centre.  It has a price range of £20-25.afé Sicilia.  Blue Spice is an average price range for a meal.  The Blue Spice coffee shop is an average priced coffee shop in the city
Decoded MR1: 
 ['name[Blue Spice], eatType[coffee shop], area[city centre]']
Decoded MR2: 
 ['

In [ ]:
# testwref_dataset = E2EDataset(testwref_df, test=True)

#Bleu Metric Calculation
raw_mrs, refs = data_retrieval(testwref_df, unique_mr=True)
raw_mrs_dataset = TestDataset(raw_mrs)
preds = batched_generate(model, tokenizer, raw_mrs_dataset)
print("1st MR's refs: ", refs[0])

bleu_score = bleu.compute(
    predictions=preds,
    references=refs
    # references=[[r] for r in refs]
)["bleu"]

print("BLEU:", bleu_score)


# #Meteor Metric Calculation
# raw_mrs, refs = data_retrieval(testwref_df, unique_mr=False)
# raw_mrs_dataset = TestDataset(raw_mrs)
# preds = batched_generate(model, tokenizer, raw_mrs_dataset)

# meteor_score = meteor.compute(
#     predictions=preds,
#     references=refs
# )["meteor"]

# print("METEOR:", meteor_score)

#0.14781731230873574,  0.47502768558940667

#24/4 night
#bleu with padding: 0.40962509428853056
#bleu without padding + with input_len trimming: 0.41153044060047933,
#bleu without padding +  with delimiter trimming: 0.4208239216046595


  0%|          | 0/630 [00:00<?, ?it/s]

1st MR's refs:  ['A coffee shop in the city centre area called Blue Spice.', 'Blue Spice is a coffee shop in city centre.']
BLEU: 0.4208239216046595


In [ ]:
# BLEU FILES
def create_bleu_files(preds_list, refs_list):
    gdir = base_dir + "/results/e2e/"
    outputs = "\n".join(preds_list)
    with open("outputs.txt", 'w') as f:
        f.writelines(outputs)

    new_refs_list = []
    ref_count = len(refs_list) - 1
    for refs in refs_list:
        new_refs = "\n".join(refs)
        new_refs_list.append(new_refs)
    with open("refs.txt", 'w') as f:
        for i, refs in enumerate(new_refs_list):
            f.writelines(refs)
            if i != ref_count:
                f.write('\n\n')


create_bleu_files(preds, refs)